# Previsão de consumo de energia de data centers com SARIMAX

Terceira etapa do trabalho. O problema é de **série temporal**: prever, hora a hora, quanta energia a **refrigeração**
(`cooling_power_kw`) de um data center vai gastar nos próximos 7 dias, a partir do histórico dela e de duas variáveis
externas (a temperatura de fora e a potência dos servidores, a "carga"), com um **SARIMAX**. O notebook também compara
os três tipos de refrigeração do dataset (ar, líquido direto no chip e híbrido com free cooling).

**Dataset:** [AI Data Center Energy Efficiency Dataset 2026](https://www.kaggle.com/datasets/eishatuzzuhra/ai-data-center-energy-efficiency-dataset-2026) (Kaggle, dados sintéticos)

**Como usar:** menu *Ambiente de execução → Executar tudo*. O notebook tenta baixar o dataset sozinho pelo `kagglehub`.
Se falhar, ele pede o upload do CSV de leituras (o arquivo com a coluna `timestamp`). Os parâmetros ajustáveis ficam na seção 2.
Para gerar os números da apresentação do site, rode `gerar_dados_apresentacao.py` (mesma configuração).

In [ ]:
!pip -q install statsmodels kagglehub

In [ ]:
import warnings, glob, os, itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

## 1. Carregar os dados

In [ ]:
def read_readings():
    """Acha o CSV de leituras (com timestamp e as potências)."""
    candidates = glob.glob("*.csv") + glob.glob("/content/*.csv")
    if not candidates:
        try:
            import kagglehub
            path = kagglehub.dataset_download(
                "eishatuzzuhra/ai-data-center-energy-efficiency-dataset-2026")
            candidates = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
            print("Baixado via kagglehub em:", path)
        except Exception as e:
            print("Download automatico falhou:", e)
    needed = {"timestamp", "total_facility_power_kw", "data_center_id"}
    for f in candidates:
        try:
            head = pd.read_csv(f, nrows=5)
        except Exception:
            continue
        if needed.issubset(head.columns):
            print("Arquivo de leituras:", f)
            return pd.read_csv(f)
    # ultimo recurso: upload manual
    from google.colab import files
    print("Envie o CSV de leituras (o que tem a coluna timestamp).")
    up = files.upload()
    for f in up.keys():
        head = pd.read_csv(f, nrows=5)
        if needed.issubset(head.columns):
            return pd.read_csv(f)
    raise FileNotFoundError("Nao achei um CSV com as colunas: " + ", ".join(sorted(needed)))


df_raw = read_readings()
print("Formato:", df_raw.shape)
display(df_raw.head())

In [ ]:
print("Linhas por data center:")
print(df_raw.groupby("data_center_id").size(), "\n")
print("data_origin:", df_raw["data_origin"].unique(), "\n")
print("Flags de qualidade:")
print(df_raw["data_quality_flag"].value_counts(dropna=False), "\n")
print("Tipos de carga (workload_type):")
print(df_raw["workload_type"].value_counts(dropna=False), "\n")
print("Valores ausentes por coluna:")
print(df_raw.isna().sum()[lambda s: s > 0])

## 2. Configuração

- `DC`: qual data center modelar (DC-01 = refrigeração a ar, DC-02 = líquido direto no chip, DC-03 = híbrido com free cooling). Usamos o DC-03, onde a temperatura de fora mais pesa.
- `TARGET`: variável a prever, a potência da refrigeração. **Não use** `total_facility_power_kw`, `pue` nem `facility_energy_kwh` como entrada: elas incluem a refrigeração (ou são calculadas a partir dela) e entregariam a resposta (vazamento de dados). A `inlet_temperature_c` também fica de fora, porque é resultado da refrigeração.
- `EXOG`: variáveis externas. São entradas do sistema, não consequências da refrigeração: a temperatura de fora e `it_power_kw` (a potência dos servidores). GPU, CPU e servidores ativos já estão resumidos em `it_power_kw`.
- `FAST`: `True` testa poucas combinações de parâmetros (rápido). `False` faz a busca completa (alguns minutos).
- `MASK_FLAGGED`: se `True`, linhas com `data_quality_flag` diferente de `ok` viram valores ausentes e são interpoladas.
- `ROLLING_WINDOWS`: quantas semanas de teste consecutivas usar na seção 11.

Se mudar algo aqui, mude também em `gerar_dados_apresentacao.py`.

In [ ]:
DC = "DC-03"
TARGET = "cooling_power_kw"
EXOG = ["ambient_temperature_c", "it_power_kw"]

FREQ = "60min"          # os dados originais sao de 15 em 15 min; agregamos por hora
PERIOD = 24             # sazonalidade diaria em dados horarios
TEST_HOURS = 24 * 7     # ultimos 7 dias viram teste
FAST = False
MASK_FLAGGED = False
ROLLING_WINDOWS = 4

ARQUITETURA = {"DC-01": "ar + chilled-water CRAH",
               "DC-02": "liquido direto no chip + ar auxiliar",
               "DC-03": "hibrido com economizer (free cooling)"}

## 3. Tratamento dos dados

Etapas: converter `timestamp` para data/hora, ordenar, remover duplicatas, achar buracos na série, interpolar valores ausentes no tempo e agregar de 15 min para 1 hora (média). A agregação reduz o ruído e deixa a sazonalidade diária com período 24, o que torna o SARIMAX muito mais rápido do que com período 96.

In [ ]:
def prepare(df_raw, dc, freq=FREQ, mask_flagged=MASK_FLAGGED):
    d = df_raw[df_raw["data_center_id"] == dc].copy()
    d["timestamp"] = pd.to_datetime(d["timestamp"], utc=True).dt.tz_localize(None)
    d = d.sort_values("timestamp")
    n_dup = int(d.duplicated("timestamp").sum())
    d = d.drop_duplicates("timestamp", keep="first").set_index("timestamp")

    cols = [TARGET] + EXOG
    flags = d["data_quality_flag"].copy()
    d = d[cols].apply(pd.to_numeric, errors="coerce")
    if mask_flagged:
        d.loc[flags.ne("ok")] = np.nan

    step = d.index.to_series().diff().mode().iloc[0]      # passo original
    full_idx = pd.date_range(d.index.min(), d.index.max(), freq=step)
    n_gap = len(full_idx) - len(d)
    d = d.reindex(full_idx)
    n_nan = int(d[cols].isna().sum().sum())
    d = d.interpolate(method="time", limit_direction="both")

    h = d.resample(freq).mean()
    info = {"dc": dc, "passo_original": str(step), "linhas_originais": len(flags),
            "duplicatas_removidas": n_dup, "timestamps_faltando": n_gap,
            "valores_nan_interpolados": n_nan, "horas_na_serie": len(h),
            "inicio": h.index.min(), "fim": h.index.max()}
    return h, info


hourly, info = prepare(df_raw, DC)
for k, v in info.items():
    print(f"{k:28s} {v}")
print()
display(hourly.describe().T)

## 4. Análise exploratória

In [ ]:
y = hourly[TARGET]
X = hourly[EXOG]

fig, ax = plt.subplots(2, 1, figsize=(13, 7))
y.plot(ax=ax[0], lw=0.8)
ax[0].set_title(f"{DC}: potencia de refrigeracao (kW), media horaria")
y.iloc[:24 * 7].plot(ax=ax[1])
ax[1].set_title("Primeira semana (zoom)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
y.groupby(y.index.hour).mean().plot(ax=ax[0], marker="o")
ax[0].set_title("Perfil medio por hora do dia (UTC)")
ax[0].set_xlabel("hora")
y.groupby(y.index.dayofweek).mean().plot(ax=ax[1], marker="o")
ax[1].set_title("Perfil medio por dia da semana (0 = segunda)")
ax[1].set_xlabel("dia")
plt.tight_layout(); plt.show()

print("Correlacao de cada variavel com o alvo:")
print(hourly.corr()[TARGET].drop(TARGET).sort_values(ascending=False).round(3))

### Comparando os três tipos de refrigeração (só olhando os dados)

- **PUE:** energia total do data center para cada 1 kW dos computadores (perto de 1 é melhor).
- **Custo de esfriar:** kW de refrigeração por kW de servidor.
- **Com carga parecida:** refrigeração por faixa de temperatura, só nas horas em que os servidores usam entre 90 e 120 kW.
- Cuidado: cada data center tem uma arquitetura, um clima e um número de servidores diferentes. Dá para ver diferenças, mas não separar as causas.

In [ ]:
d3 = df_raw.copy()
for c in ["it_power_kw", "cooling_power_kw", "ambient_temperature_c", "pue"]:
    d3[c] = pd.to_numeric(d3[c], errors="coerce")
d3 = d3.dropna(subset=["it_power_kw", "cooling_power_kw", "ambient_temperature_c", "pue"])
d3["custo_de_esfriar"] = d3["cooling_power_kw"] / d3["it_power_kw"]

resumo = d3.groupby("data_center_id").agg(
    PUE_medio=("pue", "mean"), PUE_desvio=("pue", "std"),
    custo_de_esfriar=("custo_de_esfriar", "mean"),
    refrigeracao_media_kW=("cooling_power_kw", "mean"),
    temperatura_media_C=("ambient_temperature_c", "mean"))
display(resumo.round(3))

# refrigeracao por faixa de temperatura, com carga parecida
faixas = [-30, 5, 10, 15, 20, 25, 60]
nomes = ["ate 5", "5 a 10", "10 a 15", "15 a 20", "20 a 25", "acima de 25"]
m = d3[(d3["it_power_kw"] > 90) & (d3["it_power_kw"] <= 120)].copy()
m["faixa_C"] = pd.cut(m["ambient_temperature_c"], faixas, labels=nomes)
tab = m.pivot_table(index="faixa_C", columns="data_center_id", values="cooling_power_kw", observed=False)
cont = m.pivot_table(index="faixa_C", columns="data_center_id", values="cooling_power_kw", aggfunc="count", observed=False)
tab = tab.where(cont >= 30)          # esconde faixas com poucas leituras
display(tab.round(1))
tab.plot(marker="o", title="Refrigeracao (kW) com carga parecida, por temperatura de fora (C)")
plt.show()

# quanto da variacao da refrigeracao se explica so pela carga, e pela carga + temperatura
for k, g in d3.groupby("data_center_id"):
    y_ = g["cooling_power_kw"].values
    def r2_(X):
        b = np.linalg.lstsq(X, y_, rcond=None)[0]
        return 1 - ((y_ - X @ b) ** 2).sum() / ((y_ - y_.mean()) ** 2).sum()
    um = np.ones(len(g))
    print(k, f"so carga: R2 = {r2_(np.c_[um, g.it_power_kw]):.2f} | carga + temperatura: R2 = "
             f"{r2_(np.c_[um, g.it_power_kw, g.ambient_temperature_c]):.2f}")

In [ ]:
stl = STL(y, period=PERIOD, robust=True).fit()
stl.plot(); plt.show()
Fs = max(0, 1 - stl.resid.var() / (stl.resid + stl.seasonal).var())
Ft = max(0, 1 - stl.resid.var() / (stl.resid + stl.trend).var())
print(f"Forca da sazonalidade diaria: {Fs:.2f}  (perto de 1 = muito sazonal)")
print(f"Forca da tendencia:           {Ft:.2f}")

## 5. Estacionariedade e ACF/PACF

- **ADF:** p < 0,05 sugere série estacionária.
- **KPSS:** p > 0,05 sugere série estacionária.

Se a série original não for estacionária, aplicamos diferenciação (`d`) e/ou diferenciação sazonal (`D`). Esses números orientam a escolha dos parâmetros do SARIMAX.

In [ ]:
def stationarity(s, name):
    s = s.dropna()
    adf_p = adfuller(s, autolag="AIC")[1]
    kpss_p = kpss(s, regression="c", nlags="auto")[1]
    print(f"{name:32s} ADF p={adf_p:.4f} | KPSS p={kpss_p:.4f}")

stationarity(y, "original")
stationarity(y.diff(), "d=1")
stationarity(y.diff(PERIOD), f"D=1 (lag {PERIOD})")
stationarity(y.diff(PERIOD).diff(), "D=1 e d=1")

z = y.diff(PERIOD).dropna()
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(z, lags=72, ax=ax[0], title="ACF (apos diferenciacao sazonal)")
plot_pacf(z, lags=72, ax=ax[1], title="PACF (apos diferenciacao sazonal)", method="ywm")
plt.show()

## 6. Divisão treino/teste e modelos de referência

A divisão respeita o tempo: os últimos 7 dias são o teste, sem embaralhar. As variáveis exógenas são padronizadas com média e desvio do **treino** (para não vazar informação do teste).

Os modelos de referência servem para provar que o SARIMAX acrescenta algo:
1. **Último dia repetido:** repete as últimas 24 h do treino pelos 7 dias.
2. **Perfil médio horário:** média do treino para cada hora do dia.

In [ ]:
def metrics(y_true, y_pred):
    e = np.asarray(y_true) - np.asarray(y_pred)
    return {"MAE": float(np.mean(np.abs(e))),
            "RMSE": float(np.sqrt(np.mean(e ** 2))),
            "MAPE_%": float(np.mean(np.abs(e / np.asarray(y_true))) * 100)}


def split_scale(y, X, test_hours=TEST_HOURS, end=None):
    """Treino = tudo antes da janela de teste; teste = `test_hours` horas terminando em `end`."""
    end = len(y) if end is None else end
    y_tr, y_te = y.iloc[:end - test_hours], y.iloc[end - test_hours:end]
    X_tr, X_te = X.iloc[:end - test_hours], X.iloc[end - test_hours:end]
    keep = X_tr.columns[X_tr.std() > 0]                 # tira exogena constante
    X_tr, X_te = X_tr[keep], X_te[keep]
    mu, sd = X_tr.mean(), X_tr.std()
    return y_tr, y_te, (X_tr - mu) / sd, (X_te - mu) / sd


def perfil_medio(y_tr, indice):
    perfil = y_tr.groupby(y_tr.index.hour).mean()
    return pd.Series(perfil.reindex(indice.hour).values, index=indice)


y_train, y_test, Xs_train, Xs_test = split_scale(y, X)
print(f"Treino: {len(y_train)} h ({y_train.index.min()} a {y_train.index.max()})")
print(f"Teste:  {len(y_test)} h ({y_test.index.min()} a {y_test.index.max()})")
print("Exogenas usadas:", list(Xs_train.columns))

pred_naive = pd.Series(np.tile(y_train.iloc[-PERIOD:].values, TEST_HOURS // PERIOD + 1)[:TEST_HOURS], index=y_test.index)
pred_profile = perfil_medio(y_train, y_test.index)

scores = {"Ultimo dia repetido": metrics(y_test, pred_naive),
          "Perfil medio horario": metrics(y_test, pred_profile)}
display(pd.DataFrame(scores).T.round(3))

## 7. Escolha dos parâmetros

Notação: SARIMAX(p, d, q)(P, D, Q, s), com s = 24.
- `p`, `q`: memória de curto prazo (AR e MA). Olhe o PACF e o ACF acima.
- `d`: diferenciação simples. `D`: diferenciação sazonal (1 se a sazonalidade é forte).
- `P`, `Q`: o mesmo, mas nos múltiplos de 24 horas.

A busca compara o **AIC** (menor é melhor, penaliza complexidade). Atenção: o AIC só é estritamente comparável entre modelos com a mesma diferenciação. Como D = 1 em todos, a comparação mais delicada é entre `d = 0` e `d = 1`; por isso, confirme o vencedor pelo erro no teste na seção 9.

In [ ]:
if FAST:
    orders = [(1, 0, 1), (1, 1, 1)]
    seasonals = [(0, 1, 1, PERIOD)]
else:
    orders = [(1, 0, 1), (2, 0, 1), (1, 1, 1), (2, 1, 1), (2, 0, 2)]
    seasonals = [(0, 1, 1, PERIOD), (1, 1, 1, PERIOD)]

rows = []
for o, s in itertools.product(orders, seasonals):
    t0 = time.time()
    try:
        m = SARIMAX(y_train, exog=Xs_train, order=o, seasonal_order=s,
                    enforce_stationarity=False, enforce_invertibility=False
                    ).fit(disp=False, maxiter=100)
        rows.append({"order": o, "seasonal": s, "AIC": m.aic, "BIC": m.bic,
                     "segundos": round(time.time() - t0, 1)})
    except Exception as e:
        print("Falhou", o, s, "->", e)

grid = pd.DataFrame(rows).sort_values("AIC").reset_index(drop=True)
display(grid.round(1))
best_order, best_seasonal = grid.loc[0, "order"], grid.loc[0, "seasonal"]
print("Melhor por AIC:", best_order, best_seasonal)

## 8. Modelo final e diagnóstico dos resíduos

Um bom modelo deixa resíduos parecidos com ruído branco: sem autocorrelação, média perto de zero e distribuição aproximadamente normal. No teste de Ljung-Box, **p > 0,05** indica que não sobrou autocorrelação.

In [ ]:
def fit_sarimax(y_tr, X_tr, order, seasonal):
    return SARIMAX(y_tr, exog=X_tr, order=order, seasonal_order=seasonal,
                   enforce_stationarity=False, enforce_invertibility=False
                   ).fit(disp=False, maxiter=200)


res = fit_sarimax(y_train, Xs_train, best_order, best_seasonal)
print(res.summary().tables[1])

In [ ]:
res.plot_diagnostics(figsize=(13, 8)); plt.show()

resid = res.resid.iloc[PERIOD + 1:]       # ignora o inicio, afetado pela diferenciacao
lb = acorr_ljungbox(resid, lags=[24, 48], return_df=True)
display(lb.round(4))

## 9. Previsão nos 7 dias de teste

O SARIMAX com exógenas precisa dos valores futuros das variáveis externas. Aqui usamos os valores **reais** do período de teste, o que equivale a uma previsão condicional ("dado o clima e a carga que ocorreram, qual a potência esperada?"). Na prática, esses valores viriam de previsão do tempo e de planejamento de carga. Isso deve ser dito na apresentação.

In [ ]:
fc = res.get_forecast(steps=len(y_test), exog=Xs_test)
pred_sarimax = pd.Series(fc.predicted_mean.values, index=y_test.index)
ci = fc.conf_int(alpha=0.05)
ci.index = y_test.index

# SARIMA sem exogenas, mesma ordem, so para comparar o efeito das variaveis externas
res_noexog = fit_sarimax(y_train, None, best_order, best_seasonal)
pred_sarima = pd.Series(res_noexog.get_forecast(steps=len(y_test)).predicted_mean.values,
                        index=y_test.index)

scores["SARIMA (sem exogenas)"] = metrics(y_test, pred_sarima)
scores["SARIMAX (com exogenas)"] = metrics(y_test, pred_sarimax)
tabela = pd.DataFrame(scores).T.round(3)
display(tabela)

tail = y_train.iloc[-24 * 7:]
plt.figure(figsize=(14, 5))
plt.plot(tail.index, tail, label="treino (ultima semana)", color="gray")
plt.plot(y_test.index, y_test, label="real (teste)", color="black")
plt.plot(pred_sarimax.index, pred_sarimax, label="SARIMAX", color="tab:red")
plt.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], color="tab:red", alpha=0.15,
                 label="IC 95%")
plt.title(f"{DC}: previsao de {TEST_HOURS // 24} dias da potencia de refrigeracao (kW)")
plt.legend(); plt.show()

tabela[["MAE", "RMSE"]].plot(kind="bar", figsize=(10, 4), rot=15,
                             title="Erro no teste (menor e melhor)")
plt.show()

inside = ((y_test.values >= ci.iloc[:, 0].values) & (y_test.values <= ci.iloc[:, 1].values)).mean()
print(f"Horas reais dentro do IC de 95%: {inside:.0%}  (bem calibrado = perto de 95%)")

## 10. O que cada variável externa faz

Como as exógenas foram padronizadas, o coeficiente do modelo é em kW por **desvio-padrão**. Dividindo pelo desvio-padrão do treino, obtemos kW **por unidade** da variável (por °C, por ponto percentual de GPU...). Um p-valor baixo (< 0,05) indica efeito estatisticamente relevante.

Depois, a **ablação**: refazemos o ajuste, com a mesma ordem, usando só alguns grupos de variáveis.

In [ ]:
sd_train = X.loc[y_train.index, Xs_train.columns].std()
coef = pd.DataFrame({"kW_por_desvio_padrao": res.params[Xs_train.columns],
                     "kW_por_unidade": res.params[Xs_train.columns] / sd_train,
                     "p_valor": res.pvalues[Xs_train.columns]})
display(coef.round(4))
print("Desvio-padrao de cada exogena no treino:")
print(sd_train.round(3))

In [ ]:
grupos = {"So o passado (SARIMA)": [],
          "+ clima": ["ambient_temperature_c"],
          "+ carga": ["it_power_kw"],
          "Todas as externas": list(Xs_train.columns)}
linhas = []
for nome, cols in grupos.items():
    cols = [c for c in cols if c in Xs_train.columns]
    if nome.startswith("Todas"):
        p = pred_sarimax
    elif not cols:
        p = pred_sarima
    else:
        m = fit_sarimax(y_train, Xs_train[cols], best_order, best_seasonal)
        p = pd.Series(m.get_forecast(steps=len(y_test), exog=Xs_test[cols]).predicted_mean.values, index=y_test.index)
    linhas.append({"entradas": nome, "variaveis": ", ".join(cols) or "-", **metrics(y_test, p)})
display(pd.DataFrame(linhas).round(3))

## 11. Validação em várias semanas

Uma só semana de teste pode ser sorte ou azar. Aqui repetimos treino e teste em `ROLLING_WINDOWS` semanas seguidas, sempre treinando só com o que veio antes, e comparamos com o perfil médio horário. A ordem do modelo é a escolhida na seção 7.

In [ ]:
linhas = []
n = len(y)
for w in range(ROLLING_WINDOWS):
    fim = n - (ROLLING_WINDOWS - 1 - w) * TEST_HOURS
    if fim - TEST_HOURS < 24 * 28:        # exige pelo menos 4 semanas de treino
        continue
    ytr, yte, xa, xb = split_scale(y, X, end=fim)
    m = fit_sarimax(ytr, xa, best_order, best_seasonal)
    p = pd.Series(m.get_forecast(steps=len(yte), exog=xb).predicted_mean.values, index=yte.index)
    linhas.append({"teste_de": yte.index[0].date(), "teste_ate": yte.index[-1].date(), "horas_treino": len(ytr),
                   "MAE_SARIMAX": metrics(yte, p)["MAE"], "MAE_perfil_medio": metrics(yte, perfil_medio(ytr, yte.index))["MAE"]})
janelas = pd.DataFrame(linhas)
display(janelas.round(3))
print(f"SARIMAX melhor que o perfil medio em {(janelas.MAE_SARIMAX < janelas.MAE_perfil_medio).sum()} de {len(janelas)} semanas")

## 12. Os três data centers

Repete todo o pipeline para cada data center, usando a ordem escolhida acima. É a parte que dá a "história" do trabalho: a arquitetura de refrigeração e o clima mudam a previsibilidade e o efeito da temperatura ambiente?

In [ ]:
linhas = []
for dc in sorted(df_raw["data_center_id"].unique()):
    h, _ = prepare(df_raw, dc)
    y_dc, X_dc = h[TARGET], h[EXOG]
    ytr, yte, xtr, xte = split_scale(y_dc, X_dc)
    r = fit_sarimax(ytr, xtr, best_order, best_seasonal)
    p = pd.Series(r.get_forecast(steps=len(yte), exog=xte).predicted_mean.values, index=yte.index)
    m = metrics(yte, p)
    base = metrics(yte, perfil_medio(ytr, yte.index))
    sd_dc = X_dc.loc[ytr.index, xtr.columns].std()
    linhas.append({"data_center": dc, "arquitetura": ARQUITETURA.get(dc, "-"),
                   "potencia_media_kW": round(float(y_dc.mean()), 1),
                   "MAE_SARIMAX": round(m["MAE"], 2), "RMSE_SARIMAX": round(m["RMSE"], 2),
                   "MAPE_%_SARIMAX": round(m["MAPE_%"], 2), "MAE_perfil_medio": round(base["MAE"], 2),
                   **{f"kW_por_{c}": round(float(r.params[c] / sd_dc[c]), 3) for c in xtr.columns}})
display(pd.DataFrame(linhas))

## 13. Pontos para explicar no vídeo

1. **Origem dos dados:** sintéticos (`data_origin = synthetic`). A sazonalidade tende a ser mais limpa que a de dados reais.
2. **Tratamento:** conversão de timestamp, remoção de duplicatas, interpolação de valores ausentes (`missing_sensor`), agregação de 15 min para 1 h e padronização das exógenas.
3. **Vazamento de dados:** o alvo é `cooling_power_kw`; `total_facility_power_kw`, `pue` e `facility_energy_kwh` ficaram de fora por incluírem a refrigeração ou serem calculadas a partir dela.
4. **Escolha dos parâmetros:** estacionariedade (ADF/KPSS), ACF/PACF e AIC. D = 1 por causa da sazonalidade diária forte.
5. **Validação:** divisão temporal (7 dias de teste), comparação com modelos de referência, métricas MAE, RMSE e MAPE, e repetição em várias semanas.
6. **Diagnóstico:** resíduos (Ljung-Box e gráficos).
7. **Limitações:** previsão condicional (usa exógenas reais do teste), série curta (cerca de 10 semanas, sem sazonalidade semanal explícita), efeito linear das externas e dados sintéticos.
8. **Ideias de extensão:** comparar com LSTM e trocar o clima real do teste por uma previsão do tempo.